# Embeddings by hand (Session 7)

Six short sentences about ShopLite become lists of numbers. Then we measure, by hand, how close each pair is. No vector store and no search library yet: just the embedding model and numpy.

In [1]:
from sentence_transformers import SentenceTransformer

# Runs on this laptop: no key, no cost per call.
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

sentences = [
    "How much does shipping cost?",
    "Shipping is free on orders of 999 rupees or more.",
    "def shipping_rate_for(amount_paise, pincode):",
    "Only one coupon can be used per order.",
    "A coupon takes money off the order at checkout.",
    "The warehouse keeps a history of every stock change.",
]
vectors = model.encode_document(sentences)
print(vectors.shape)  # six sentences, each a list of 1,024 numbers

/Users/codingninjas/Desktop/Learning-Architecture-Studio/Courses/applied-ai-engineering-bootcamp/05_projects/askmyrepo/repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1118 23:28:35.355000 22349 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/Qwen/Qwen3-Embedding-0.6B: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'Qwen/Qwen3-Embedding-0.6B'.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 17883.94it/s]

(6, 1024)


Each sentence is now 1,024 numbers. Here are the first five numbers of the first one: on their own they mean nothing; only distances between vectors mean something.

In [2]:
print(vectors[0][:5])

[-0.0559082  -0.00524902 -0.00141144 -0.01495361 -0.01660156]


## Cosine similarity, by hand

Cosine similarity = (a · b) / (|a| × |b|): multiply the two lists number by number and add up, then divide by both lengths. 1 means the same direction; near 0 means unrelated.

In [3]:
import numpy as np


def cosine(a, b):
    """How close two vectors point: (a . b) / (|a| x |b|)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print(round(cosine(vectors[0], vectors[1]), 3))  # a shipping question and a shipping fact
print(round(cosine(vectors[0], vectors[5]), 3))  # a shipping question and the warehouse

0.623
0.341


## Every pair, closest first

In [4]:
pairs = []
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        pairs.append((cosine(vectors[i], vectors[j]), sentences[i], sentences[j]))
for score, first, second in sorted(pairs, reverse=True):
    print(f"{score:.3f}  {first}  |  {second}")

0.741  Only one coupon can be used per order.  |  A coupon takes money off the order at checkout.
0.623  How much does shipping cost?  |  Shipping is free on orders of 999 rupees or more.
0.620  Shipping is free on orders of 999 rupees or more.  |  def shipping_rate_for(amount_paise, pincode):
0.575  How much does shipping cost?  |  def shipping_rate_for(amount_paise, pincode):
0.492  Shipping is free on orders of 999 rupees or more.  |  A coupon takes money off the order at checkout.
0.455  Shipping is free on orders of 999 rupees or more.  |  Only one coupon can be used per order.
0.429  def shipping_rate_for(amount_paise, pincode):  |  A coupon takes money off the order at checkout.
0.424  A coupon takes money off the order at checkout.  |  The warehouse keeps a history of every stock change.
0.422  def shipping_rate_for(amount_paise, pincode):  |  Only one coupon can be used per order.
0.397  How much does shipping cost?  |  A coupon takes money off the order at checkout.
0.366  On

## A question finds its answer

A question is embedded with the model's question instruction (`encode_query`), passages without it (`encode_document`). Then the closest passage is the one to read.

In [5]:
question = model.encode_query(["Is there a limit on coupons?"])[0]
ranked = sorted(zip(sentences, vectors, strict=True), key=lambda p: -cosine(question, p[1]))
for sentence, vector in ranked:
    print(f"{cosine(question, vector):.3f}  {sentence}")

0.662  Only one coupon can be used per order.
0.555  A coupon takes money off the order at checkout.
0.288  Shipping is free on orders of 999 rupees or more.
0.276  def shipping_rate_for(amount_paise, pincode):
0.232  How much does shipping cost?
0.193  The warehouse keeps a history of every stock change.
